# 1. Compliance Fundamentals

In this notebook you'll learn the three tools Microsoft gives you to *manage* compliance:

1. **Service Trust Portal (STP)** - proof that *Microsoft's* cloud is compliant.
2. **Compliance Manager** - a score that tracks *your organization's* compliance work.
3. **Microsoft Priva** - handles privacy risk and data subject requests (GDPR right-to-access, right-to-erasure).

By the end you'll be able to:

- Explain the shared responsibility model for compliance.
- Compute a compliance score from improvement actions.
- Walk a GDPR subject rights request through its lifecycle.

## Setup

This lab runs entirely in Python - no Docker, no cloud subscription needed. All Microsoft Purview features are *simulated* in code so you can explore the concepts without a real Microsoft 365 tenant.

**Before you run any cell**:

1. From the lab folder, install dependencies:
   ```bash
   cd security-certs/sc-900/04-compliance-and-purview
   uv sync
   ```
2. In VS Code, open this notebook and pick the **`.venv` kernel** from the kernel picker (top-right of the notebook).
3. If the `.venv` kernel doesn't show up, reload the VS Code window: `Cmd+Shift+P` -> *Developer: Reload Window*.

> 💡 Everything in this notebook is a *simulation*. The goal is to build intuition for what Purview does - then you'll recognize it instantly when you see the real portal.

---
## Service Trust Portal (STP)

**URL**: [servicetrust.microsoft.com](https://servicetrust.microsoft.com)

Think of STP as *Microsoft's filing cabinet of compliance certificates*. When an auditor says *"Prove Azure is SOC 2 compliant,"* you download the SOC 2 report from the STP and hand it over.

| Section | What you'll find |
|---------|------------------|
| **Certifications & Regulations** | Audit reports (SOC 1/2/3, ISO 27001, FedRAMP, PCI DSS, HIPAA) |
| **Reports & Whitepapers** | Pen test results, privacy docs, security assessments |
| **Industry resources** | Guidance for healthcare, financial services, government |
| **Compliance resources** | Regional info (EU, US, Asia) |

### Exam tip

- STP = **Microsoft's** compliance evidence for *their* services.
- **YOUR** organization's compliance is tracked in **Compliance Manager** (next section).

---
## Shared responsibility for compliance

Compliance is never 100% Microsoft's job. Some controls are Microsoft's, some are yours, some are shared.

| Responsibility | Example | Who owns it |
|---|---|---|
| Physical datacenter security | Biometric locks, 24/7 guards | Microsoft |
| Encryption of data at rest | Storage-level AES-256 | Microsoft |
| Who gets admin access to *your* tenant | RBAC, Conditional Access | **You** |
| Classifying *your* sensitive data | Applying sensitivity labels | **You** |
| Incident response to *your* compromised users | SOC playbooks | **You** |

The *shared* controls (like identity) are the ones beginners usually get wrong - Microsoft builds the feature, but you have to turn it on and configure it.

## Microsoft's privacy principles

Microsoft publishes six privacy principles - memorize them, they come up on the exam:

1. **Control** - you control your data.
2. **Transparency** - Microsoft is clear about what data it collects and why.
3. **Security** - strong encryption + security by design.
4. **Strong legal protections** - Microsoft respects local privacy laws, challenges overbroad government requests.
5. **No content-based targeting** - your email/chat/files are **not** used for ads.
6. **Benefits to you** - any data collected must improve your experience.

---
## Microsoft Purview Compliance Manager

Compliance Manager helps you **measure and improve** compliance across many regulations at once.

### Core vocabulary

| Term | Meaning (plain English) |
|---|---|
| **Assessment** | Your scorecard for one regulation (e.g. GDPR) |
| **Control** | One requirement inside that regulation (e.g. "encrypt PII") |
| **Improvement action** | A concrete task you do to satisfy a control |
| **Microsoft-managed action** | Already done by Microsoft - free points |
| **Customer-managed action** | Your homework |
| **Compliance score** | Weighted % of completed improvement actions |

### Bad vs. Best: tracking compliance

Below we compare two ways of tracking GDPR + ISO 27001 posture:

- **BAD**: a spreadsheet of controls, maintained by hand. No priority, no ownership, easy to forget.
- **BEST**: Compliance Manager-style scoring - each action has a point value so you know what to fix *first*.

In [ ]:
# BAD: unstructured spreadsheet approach
# A list of controls with no priority, no ownership, no score.
spreadsheet = [
    'GDPR: need DSR process - not sure who owns',
    'GDPR: right to erasure - done (I think?)',
    'GDPR: 72-hour breach notification - TODO',
    'ISO: access control policy - done',
    'ISO: incident management - TODO',
]
print('=== BAD: compliance-as-a-spreadsheet ===')
for line in spreadsheet:
    print(' -', line)
print('\nProblem: no score, no priority, no idea what to tackle first.')

In [ ]:
# BEST: Compliance Manager-style structured assessments with scoring
ASSESSMENTS = [
    {
        'regulation': 'GDPR',
        'controls': [
            {'name': 'Data subject access request process', 'score': 10, 'status': 'implemented', 'owner': 'microsoft'},
            {'name': 'Right to erasure', 'score': 10, 'status': 'implemented', 'owner': 'customer'},
            {'name': 'Data breach notification (72 hours)', 'score': 15, 'status': 'not_started', 'owner': 'customer'},
            {'name': 'Data protection impact assessment', 'score': 10, 'status': 'in_progress', 'owner': 'customer'},
            {'name': 'Encryption of personal data', 'score': 15, 'status': 'implemented', 'owner': 'shared'},
        ],
    },
    {
        'regulation': 'ISO 27001',
        'controls': [
            {'name': 'Access control policy', 'score': 8, 'status': 'implemented', 'owner': 'customer'},
            {'name': 'Cryptographic controls', 'score': 8, 'status': 'implemented', 'owner': 'microsoft'},
            {'name': 'Incident management', 'score': 12, 'status': 'not_started', 'owner': 'customer'},
            {'name': 'Business continuity', 'score': 10, 'status': 'in_progress', 'owner': 'customer'},
        ],
    },
]

print('=== BEST: Microsoft Purview Compliance Manager ===\n')
total_max = total_earned = 0
open_actions = []

for assessment in ASSESSMENTS:
    max_score = sum(c['score'] for c in assessment['controls'])
    earned = sum(c['score'] for c in assessment['controls'] if c['status'] == 'implemented')
    total_max += max_score
    total_earned += earned
    pct = (earned / max_score) * 100
    print(f'[Assessment] {assessment["regulation"]}: {earned}/{max_score} ({pct:.0f}%)')
    for c in assessment['controls']:
        icon = {'implemented': 'OK ', 'in_progress': '.. ', 'not_started': '   '}[c['status']]
        print(f'   [{icon}] [{c["owner"]:<9}] {c["name"]:<45} (+{c["score"]} pts)')
        if c['status'] != 'implemented' and c['owner'] != 'microsoft':
            open_actions.append((c['score'], assessment['regulation'], c['name']))
    print()

overall = (total_earned / total_max) * 100
print(f'Overall Compliance Score: {total_earned}/{total_max} ({overall:.0f}%)')

print('\nTop improvement actions (highest impact first):')
for score, reg, name in sorted(open_actions, reverse=True)[:3]:
    print(f'   +{score} pts -> [{reg}] {name}')

### Exam tip

- Compliance Manager gives you a **compliance score** and **improvement actions**.
- It's **not** a guarantee of compliance - it's a risk-based *measurement* tool. Microsoft says so explicitly, and the exam tests that distinction.
- Microsoft-managed actions are already completed by Microsoft (free points). **Customer-managed** actions are your homework.
- The real score is **risk-weighted**, not a flat sum: an action is worth more if it is *preventative* (rather than detective or corrective) and *mandatory* (rather than discretionary). Our simulation above uses flat points to keep the code readable - the shape of the answer is what matters.
- Compliance Manager lives in the **Microsoft Purview portal**; the Service Trust Portal is a separate site for *Microsoft's own* audit evidence.

---
## Microsoft Priva - privacy risk & subject rights

Priva sits on top of Microsoft 365 and focuses on **personal data** specifically:

| Feature | What it does | Real-world use case |
|---|---|---|
| **Privacy Risk Management** | Finds personal data, flags risky handling (overexposure, transfers, hoarding) | Detect an HR spreadsheet shared with the whole company |
| **Subject Rights Requests (SRR)** | Automates GDPR/CCPA "right to access/delete" requests | EU customer emails "delete my data"; Priva finds and packages everything |

### The SRR lifecycle

```
1. Receive request -> customer asks "what do you have on me?"
2. Locate data     -> Priva searches Exchange, SharePoint, OneDrive, Teams
3. Review          -> reviewer approves/removes items (legal hold, privilege)
4. Respond         -> export package or confirm deletion within statutory deadline
```

GDPR (Article 12) gives you **one month** to respond — commonly taught as **30 days** — extendable by **two further months** for complex or numerous requests, provided you tell the data subject why within the first month. Below we simulate that pipeline.

In [ ]:
from datetime import datetime, timedelta

# Simulated data store across M365 workloads.
TENANT_DATA = [
    {'workload': 'Exchange',   'id': 'mail-1', 'subject': 'Order confirmation', 'contains': ['alice@example.com', 'credit card']},
    {'workload': 'Exchange',   'id': 'mail-2', 'subject': 'Newsletter signup',  'contains': ['alice@example.com']},
    {'workload': 'SharePoint', 'id': 'doc-1',  'subject': 'CRM export Q1',      'contains': ['alice@example.com', 'phone: 555-0101']},
    {'workload': 'OneDrive',   'id': 'doc-2',  'subject': 'Marketing list',     'contains': ['bob@example.com']},
    {'workload': 'Teams',      'id': 'msg-1',  'subject': 'Chat with support',  'contains': ['alice@example.com']},
]

def subject_rights_request(data_subject_email: str, request_type: str) -> None:
    received = datetime(2026, 4, 21)
    deadline = received + timedelta(days=30)
    print(f'[SRR] received for {data_subject_email} - type: {request_type}')
    print(f'   Received: {received:%Y-%m-%d}   Deadline: {deadline:%Y-%m-%d}\n')

    print('Step 1 - locate all personal data across M365:')
    hits = [item for item in TENANT_DATA if any(data_subject_email in c for c in item['contains'])]
    for h in hits:
        print(f'   - [{h["workload"]:<10}] {h["id"]}: "{h["subject"]}"')
    print(f'   -> {len(hits)} items found.\n')

    print('Step 2 - reviewer decides what to include (skip privileged/hold items):')
    approved = [h for h in hits if 'credit card' not in h['contains']]
    for h in approved:
        print(f'   [include] {h["id"]}')
    for h in hits:
        if h not in approved:
            print(f'   [redact ] {h["id"]} (contains payment data)')
    print()

    print(f'Step 3 - respond to subject ({request_type}):')
    if request_type == 'access':
        print(f'   Export ZIP with {len(approved)} items delivered to {data_subject_email}.')
    elif request_type == 'erasure':
        print(f'   Deleted {len(approved)} items; {len(hits) - len(approved)} retained under legal hold.')

subject_rights_request('alice@example.com', 'access')
print('\n' + '-' * 60 + '\n')
subject_rights_request('alice@example.com', 'erasure')

---
## Summary

| Concept | Key fact |
|---------|----------|
| **Service Trust Portal** | Microsoft's audit reports and compliance docs |
| **Shared responsibility** | Microsoft secures the cloud; you secure what's *in* it |
| **Privacy principles** | Control, transparency, security, legal protections, no content targeting, benefits |
| **Priva** | Privacy risk management + subject rights requests in M365 |
| **Compliance Manager** | Measure compliance with a score + improvement actions |
| **Compliance score** | Percentage based on completed improvement actions |
| **SRR deadline (GDPR)** | One month (~30 days), extendable by two further months for complex requests |

**Next**: [Notebook 2 - Information Protection and DLP](02_information_protection_and_dlp.ipynb)

---
## Self-check — compliance fundamentals

In [ ]:
QUIZ = [
    {'id': 'Q1',
     'q': 'An auditor asks for the ISO 27001 certificate covering Azure datacentres. Where do you get it?',
     'options': {'A': 'Microsoft Purview Compliance Manager', 'B': 'Service Trust Portal',
                 'C': 'Microsoft Defender for Cloud', 'D': 'Microsoft Priva'},
     'a': 'B',
     'why': 'The Service Trust Portal publishes MICROSOFT\'s audit reports and certifications. Compliance Manager '
            'measures YOUR organisation\'s posture. Remember the split: STP = evidence about Microsoft; '
            'Compliance Manager = evidence about you.'},
    {'id': 'Q2',
     'q': 'Which of these is a Microsoft privacy principle?',
     'options': {'A': 'Zero Trust', 'B': 'No content-based targeting',
                 'C': 'Least privilege', 'D': 'Assume breach'},
     'a': 'B',
     'why': 'The six privacy principles are Control, Transparency, Security, Strong legal protections, No '
            'content-based targeting, and Benefits to you. The other three options are Zero Trust principles - '
            'a classic mix-and-match distractor.'},
    {'id': 'Q3',
     'q': 'A European customer emails "send me everything you hold about me". Which Microsoft solution automates '
          'finding and packaging that data?',
     'options': {'A': 'Microsoft Priva Subject Rights Requests', 'B': 'Microsoft Purview DLP',
                 'C': 'Microsoft Purview Audit', 'D': 'Compliance Manager'},
     'a': 'A',
     'why': 'Priva handles privacy risk and subject rights requests (access, export, erasure) across Microsoft 365. '
            'DLP prevents leaks, Audit records activity, Compliance Manager scores controls.'},
    {'id': 'Q4',
     'q': '"Our EU customer data must be physically stored in the EU." What concept is this?',
     'options': {'A': 'Data sovereignty', 'B': 'Data residency', 'C': 'Data classification', 'D': 'Data lifecycle management'},
     'a': 'B',
     'why': 'Residency = WHERE the bytes sit (you pick the Azure region). Sovereignty = WHOSE LAWS apply to them '
            'once they sit there. The exam pairs these two constantly.'},
    {'id': 'Q5',
     'q': 'In Compliance Manager, an improvement action is marked "Microsoft-managed". What does that mean?',
     'options': {'A': 'You must implement it and Microsoft will audit you',
                 'B': 'Microsoft has already implemented it; the points are credited to you',
                 'C': 'It cannot be scored', 'D': 'It only applies to Azure, not Microsoft 365'},
     'a': 'B',
     'why': 'Microsoft-managed actions are controls Microsoft implements in its own infrastructure. They are '
            'already complete, so they contribute points without work from you. Customer-managed actions are yours.'},
]

MY_ANSWERS = {'Q1': 'B', 'Q2': 'B', 'Q3': 'A', 'Q4': 'B', 'Q5': 'B'}

score = 0
for q in QUIZ:
    mine = MY_ANSWERS.get(q['id'], '').strip().upper()
    ok = mine == q['a']
    score += ok
    print(f'{"PASS" if ok else "FAIL"}  {q["id"]}: {q["q"]}')
    for k, v in q['options'].items():
        print(f'         {k}. {v} {"<-- correct" if k == q["a"] else ""}')
    print(f'         your answer: {mine or "(blank)"}')
    print(f'         why: {q["why"]}\n')
print(f'Score: {score}/{len(QUIZ)}')
